# Day 4B — SHAP Explainability

Explain the **saved Day 3 XGBoost churn model** with SHAP.

- Do **not** retrain the model
- SHAP shows feature **contribution to the model prediction**
- SHAP does **not** prove causation

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from churn_model import load_churn_model, load_preprocessor, predict_churn
from preprocessing import create_features, get_feature_matrix

FIGURES = PROJECT_ROOT / "reports" / "figures"
REPORTS = PROJECT_ROOT / "reports"
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load Day 3 model + preprocessor

In [ ]:
model = load_churn_model()
preprocessor = load_preprocessor()
meta = joblib.load(PROJECT_ROOT / "models" / "churn_model_metadata.pkl")
print("Best model from Day 3:", meta.get("best_model_name"))
print("Python class:", type(model).__name__)
print("Using shap.TreeExplainer because the saved model is tree-based (XGBoost).")

## 2. Prepare model-input features (same Day 2 preprocessing)

In [ ]:
cleaned = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_telco.csv")
featured = create_features(cleaned)
X, y = get_feature_matrix(featured)
feature_names = list(preprocessor.get_feature_names_out())
X_proc = pd.DataFrame(preprocessor.transform(X), columns=feature_names)
preds = predict_churn(model, X_proc)

meta_df = cleaned.copy().reset_index(drop=True)
meta_df["churn_prediction"] = preds["prediction"]
meta_df["churn_probability"] = preds["probability"]
meta_df["risk_level"] = preds["risk_level"]

print("Model input shape:", X_proc.shape)
print("Feature count:", len(feature_names))

## 3. Global SHAP (sample for speed)

In [ ]:
rng = np.random.RandomState(42)
n = min(500, len(X_proc))
idx = rng.choice(len(X_proc), size=n, replace=False)
X_sample = X_proc.iloc[idx]
sample_meta = meta_df.iloc[idx].reset_index(drop=True)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)
shap_matrix = shap_values[1] if isinstance(shap_values, list) else shap_values
assert shap_matrix.shape[1] == X_sample.shape[1]
print("SHAP shape:", shap_matrix.shape)

shap.summary_plot(shap_matrix, X_sample, show=False, max_display=20)
plt.tight_layout(); plt.savefig(FIGURES/"shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()

shap.summary_plot(shap_matrix, X_sample, plot_type="bar", show=False, max_display=20)
plt.tight_layout(); plt.savefig(FIGURES/"shap_feature_importance.png", dpi=150, bbox_inches="tight"); plt.show()

importance = pd.DataFrame({
    "feature": X_sample.columns,
    "mean_absolute_shap_value": np.abs(shap_matrix).mean(axis=0),
}).sort_values("mean_absolute_shap_value", ascending=False).reset_index(drop=True)
importance.to_csv(REPORTS/"shap_feature_importance.csv", index=False)
display(importance.head(15))

## 4. Individual customer explanation

In [ ]:
high_idx = sample_meta.index[sample_meta["risk_level"] == "High"]
local_i = int(high_idx[0]) if len(high_idx) else 0
customer = sample_meta.iloc[local_i]
local_shap = shap_matrix[local_i]

local_df = pd.DataFrame({
    "feature": X_sample.columns,
    "shap_value": local_shap,
    "feature_value": X_sample.iloc[local_i].values,
})
local_df["abs_shap"] = local_df["shap_value"].abs()
local_df = local_df.sort_values("abs_shap", ascending=False)

print("Customer ID:", customer["customerID"])
print("Actual Churn:", customer["Churn"])
print("Predicted Churn:", customer["churn_prediction"])
print(f"Churn Probability: {customer['churn_probability']:.4f}")
print("Risk Level:", customer["risk_level"])
print("\nTop factors toward churn:")
display(local_df[local_df["shap_value"] > 0].head(5))
print("Top factors toward stay:")
display(local_df[local_df["shap_value"] < 0].head(5))
print("\nThese features contributed most strongly to this customer's predicted churn risk.")
print("This is model contribution/association — not proof of causation.")

top_local = local_df.head(12).sort_values("shap_value")
colors = ["#d62728" if v > 0 else "#2ca02c" for v in top_local["shap_value"]]
plt.figure(figsize=(8,6))
plt.barh(top_local["feature"], top_local["shap_value"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"SHAP contributions — {customer['customerID']}")
plt.xlabel("SHAP value (positive = toward churn)")
plt.tight_layout(); plt.savefig(FIGURES/"shap_individual_explanation.png", dpi=150, bbox_inches="tight"); plt.show()

pd.DataFrame({
    "customerID": customer["customerID"],
    "actual_churn": customer["Churn"],
    "predicted_churn": customer["churn_prediction"],
    "churn_probability": customer["churn_probability"],
    "risk_level": customer["risk_level"],
    "feature": local_df["feature"],
    "shap_value": local_df["shap_value"],
    "feature_value": local_df["feature_value"],
}).to_csv(REPORTS/"shap_individual_explanation.csv", index=False)

## 5. Business insights (from this run's actual SHAP table)

Read the top rows of `shap_feature_importance.csv` above.
Combine with Day 4A retention tables for high-risk / high-LTV prioritization.